# Ordered Logistic Regression Results Dataset Exploration with `mlcroissant`
This notebook provides a practical guide for loading, exploring, and performing basic analysis on the FAIR\^2 dataset (ordered logistic regression results for adoption predictors of indigenous and modern knowledge in rangeland management practices, Northern Kenya) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema JSON-LD file.

In [ ]:
# Ensure `mlcroissant` is installed (run in your Jupyter environment if needed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a mlcroissant.DatasetMetadata object

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All Croissant entities should be referenced by their `@id` for transparency and code reproducibility.

Let's list all available record sets and their contents.

In [ ]:
# List all record sets and their fields by @id
record_sets = metadata.record_sets

if not record_sets:
    print("No record sets declared in this dataset's top-level metadata.\nAttempting to list by dataset.records()...")
    # Try retrieving records: mlcroissant may still load available tables.
    possible_record_sets = dataset.record_sets
    print(f"Available record_sets found via mlcroissant dataset interface:\n")
    for rs in possible_record_sets:
        print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name', '(no name)')}")
else:
    print("Declared record sets in Croissant metadata:")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs.id} | Name: {rs.name}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    • Field @id: {field.id} | Name: {field.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the actual record set `@id`s from the overview above. For demonstration, we'll extract all available record sets using their `@id` keys.

In [ ]:
# Find all record set @ids via dataset.record_sets (returns a list of dicts)
record_sets_all = dataset.record_sets
record_set_ids = [rs['@id'] for rs in record_sets_all]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if len(records) == 0:
            print(f"No records found for record_set {record_set_id}.")
            continue
        # Note: the columns will be keys of the records dict
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record_set {record_set_id} with columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load records for record_set {record_set_id}: {e}")

# For demonstration, let's pick the first available record set (if any)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nMain record set selected: {main_record_set_id}")
    print("Columns available:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No data available in any record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps (e.g., filtering by a numeric field, normalizing, grouping). We demonstrate this on the primary record set extracted above and reference all columns by their Croissant `@id`.

**Replace `numeric_field_id` and `group_field_id` with suitable `@id`s from the previous cell output, based on actual loaded columns.**

In [ ]:
# Example: EDA - Filtering, Normalization, Grouping

# Pick the main dataframe and list its columns
if dataframes:
    df = dataframes[main_record_set_id]
    print("All columns:", list(df.columns))
    # You may need to list columns to choose appropriate numeric/group fields by their @id

    # -------
    # For demonstration, pick a numeric field by @id (update to a real @id from your columns)
    numeric_field = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else None
    if numeric_field is None:
        print("No numeric fields found in this record set.")
    else:
        print(f"Using numeric field @id: {numeric_field}")
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records (with {numeric_field} > {threshold}): {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a categorical field (pick a string/object column)
        string_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field = None
        for c in string_cols:
            if c != numeric_field and c != norm_col:
                group_field = c
                break
        if group_field:
            print(f"\nGrouping by field @id: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_of_{numeric_field}")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's visualize a numeric field's distribution and, if possible, compare across a grouping field, referencing all variables by their Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of numeric field (@id: {numeric_field})")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group_field (if available)
    if group_field:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR\^2 Croissant dataset using `mlcroissant`, referencing all dataset structures via their Croissant `@id`. You can adapt this notebook to analyze individual record sets and fields further, depending on your project or research goals.

**Key takeaways:**
- Always reference Croissant entities (record sets, fields, columns) by their `@id` for reproducibility.
- The `mlcroissant` package allows seamless integration between dataset metadata and underlying data.
- The approach demonstrated here can be adapted for any Croissant-described dataset.